# Stage 2 Validate Result Findings of ZTF with our own findings from Rungs 1-3

## Cell 1: query IRSA for our field's products

In [ ]:
import sys
from pathlib import Path

# Bootstrap: put the project root on sys.path so `import config` resolves.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import config  # sets $ZTFDATA + the np.in1d shim (import-first rule for ztfquery)
from ztfquery import query

# Query the SAME field/ccd/qid/filter as our science frame (ztf_20180322273264).
zquery = query.ZTFQuery()
zquery.load_metadata(
    radec=[150, 2],
    size=0.01,
    sql_query="field=468 and ccdid=3 and qid=2 and fid=2",  # fid=2 = zr filter
)

meta = zquery.metatable
print(f"Found {len(meta)} matching rows.")
print("Columns available:", list(meta.columns))
print()
print(meta[['obsjd', 'field', 'ccdid', 'qid', 'filtercode']].head(10))



## Cell 2: Find the row matching OUR science frame's epoch

In [ ]:
target_ffd = 20180322273264

# Some ztfquery versions store filefracday as int, some as str — match robustly.
mask = meta['filefracday'].astype('int64') == target_ffd
match = meta[mask]

print(f"Rows matching filefracday={target_ffd}: {len(match)}")
if len(match):
    cols = ['filefracday', 'field', 'ccdid', 'qid', 'filtercode', 'obsjd', 'seeing', 'infobits']
    print(match[cols].to_string())
else:
    # Fallback: show the filefracday values we DO have, to spot the right one
    print("No exact match. Available filefracday values (first 15):")
    print(sorted(meta['filefracday'].astype('int64').unique())[:15])


## Cell 3: actually downloading ZTF's official difference imaging software for exposure

In [ ]:
zquery.download_data(
    "scimrefdiffimg.fits.fz",
    indexes=[772],
    show_progress=True,
)

print("Download attempted. locating the file ...")

#Find what landed on disk
import glob
from ztfquery.io import LOCALSOURCE
diff_files = glob.glob(str(Path(LOCALSOURCE) / "**" / "*scrimrefdiffimg"), recursive=True)
for f in diff_files:
    print(" ", f)
    print(f"\n Found {len(diff_files)} difference file(s).")

## Cell 4: Load ZTF's official imaging now

In [ ]:
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np 
import matplotlib.pyplot as plt 

ztf_diff_path = (Path(LOCALSOURCE) / "sci" / "2018" / "0322" / "273264"
                 / "ztf_20180322273264_000468_zr_c03_o_q2_scimrefdiffimg.fits.fz")

with fits.open(ztf_diff_path) as hdul:
    hdul.info()
    hdu = hdul[1] if hdul[0].data is None else hdul[0]
    ztf_diff_full = hdu.data.astype(float)
    ztf_diff_wcs = WCS(hdu.header)

print(f"\nZTF official diff: shape={ztf_diff_full.shape}, "
      f"WCS celestial={ztf_diff_wcs.has_celestial}")

#actually view what has been computed
finite = ztf_diff_full[np.isfinite(ztf_diff_full)]
lim = float(np.nanpercentile(np.abs(finite), 99))
plt.figure(figsize=(8,8))
plt.imshow(ztf_diff_full, cmap="gray", origin="lower", vmin=-lim, vmax=+lim)
plt.title("ZTF OFFICIAL difference image (full frame)", fontsize=10)
plt.colorbar(label="difference (centered on 0)")
plt.show()

## Cell 5: crop ZTF's official diff to our central 1000x1000 patch

In [ ]:
from operator import pos
from astropy.nddata import Cutout2D

size = (1000, 1000)
center = (ztf_diff_full.shape[1] // 2, ztf_diff_full.shape[0] // 2)
ztf_cut = Cutout2D(ztf_diff_full, position=center, size=size, wcs=ztf_diff_wcs)
ztf_diff = ztf_cut.data

finite = ztf_diff[np.isfinite(ztf_diff)]
lim = float(np.nanpercentile(np.abs(finite), 99))
ztf_std = float(np.nanstd(ztf_diff))

plt.figure(figsize=(7,7))
plt.imshow(ztf_diff, cmap="gray", origin="lower", vmin=-lim, vmax=+lim)
plt.title(f"ZTF OFFICIAL difference — central 1000x1000\nstd={ztf_std:.1f}", fontsize=10)
plt.show()

print(f"ZTF official diff (central patch): std = {ztf_std:.3f}")
print("This is the GOLD STANDARD: flat gray + sparse clean residual points.")
print("Compare by eye to your ois result (diff_ois) in stage2_subtraction.ipynb.")


## Cell 6: save in file on computer for Stage 3 configuration 

In [ ]:
# Stage 2 → Stage 3 handoff: save ZTF's official difference image to a clean folder.
# New folder ztfdata/difference/ (sibling of ztfdata/aligned/), full frame + WCS preserved.
from pathlib import Path
from astropy.io import fits

diff_dir = Path(LOCALSOURCE) / "difference"
diff_dir.mkdir(parents=True, exist_ok=True)

# Name it clearly as the ZTF-official difference for this exposure.
out_path = diff_dir / "ztf_20180322273264_000468_zr_c03_o_q2_official_diff.fits"

# Write the FULL-FRAME difference array with its WCS in the header (so sources can later
# be mapped to RA/Dec). ztf_diff_full + ztf_diff_wcs were created in the load cell (Cell 4).
fits.PrimaryHDU(
    data=ztf_diff_full,
    header=ztf_diff_wcs.to_header(),
).writeto(out_path, overwrite=True)

print(f"Saved ZTF official difference -> {out_path}")
print(f"  shape = {ztf_diff_full.shape}, WCS celestial = {ztf_diff_wcs.has_celestial}")
